In [7]:
!pip install PyPDF2 groq

In [8]:
import os
from groq import Groq
from google.colab import userdata
sec_key=userdata.get("GROQ_API_KEY")

os.environ['GROQ_API_KEY'] = sec_key

client = Groq()

In [10]:
import PyPDF2
from pathlib import Path
import re
import json


class ATSScorer:
    def __init__(self, api_key):
        self.client = Groq(api_key=api_key)

    def extract_pdf_text(self, pdf_path):
        """Extract text from a single PDF file"""
        try:
            pdf_path = Path(pdf_path)
            if not pdf_path.exists() or pdf_path.suffix.lower() != '.pdf':
                raise ValueError(f"Invalid PDF file path: {pdf_path}")
            with open(pdf_path, 'rb') as file:
                pdf_reader = PyPDF2.PdfReader(file)
                text = ""
                for page in pdf_reader.pages:
                    page_text = page.extract_text()
                    if page_text:
                        text += page_text + "\n"
                text = re.sub(r'\s+', ' ', text).strip()
                if not text:
                    raise ValueError(f"No text could be extracted from {pdf_path}")
                return text
        except Exception as e:
            print(f"Error extracting text from PDF {pdf_path}: {str(e)}")
            return None

    def process_resume_folder(self, folder_path):
        """Process all PDFs in a folder and return a dictionary of extracted texts"""
        folder_path = Path(folder_path)
        if not folder_path.is_dir():
            raise ValueError(f"{folder_path} is not a valid directory")
        resume_texts = {}
        for pdf_file in folder_path.glob("*.pdf"):
            text = self.extract_pdf_text(pdf_file)
            resume_texts[pdf_file.name] = text
        return resume_texts

    def validate_json_response(self, data):
        """Validate that the response contains all required fields"""
        required_fields = {
            "Candidate Name": str, "Mail ID": str, "Phone Number": str, "Address": str,
            "Job Role": str, "Nationality": str, "Contract Type": str,
            "Education": list, "Work Experience": list, "ATS Score": int,
            "Missing Keywords": list, "Profile Summary": dict
        }
        if not isinstance(data, dict):
            return False
        return all(field in data and isinstance(data[field], field_type)
                  for field, field_type in required_fields.items())

    def calculate_ats_score(self, resume_text, job_description):
        """Calculate ATS score and generate detailed analysis using Groq LLM"""
        if not resume_text or not job_description:
            return self._error_response("Missing resume or job description")

        prompt = f"""
        You are an advanced ATS (Applicant Tracking System) tasked with evaluating resumes against a job description. Your goal is to assess alignment based on skills, qualifications, and experience, and assign an accurate ATS score with detailed feedback.

        ### Steps
        1. **Parse Inputs**: Extract from job description (JD) and resume:
            - Job Role
            - Required Skills
            - Qualifications (education, certifications)
            - Work Experience (years, roles, responsibilities)
        2. **Evaluate Resume**: Compare resume to JD:
            - Match skills, qualifications, and experiences
            - Identify missing keywords or requirements
        3. **Calculate ATS Score**: Assign a score (0-100, no decimals) based on:
            - Skills Match: 40% (exact matches of required skills)
            - Experience Match: 30% (years and relevance of experience)
            - Qualifications Match: 20% (education/certifications)
            - Keyword Presence: 10% (JD-specific terms)
            - Deduct points for missing critical requirements
        4. **Profile Summary**: Provide:
            - Detailed justification of the score
            - Breakdown showing points awarded/deducted per category
        5. **Extract Core Information**: From resume:
            - Candidate Name, Mail ID, Phone Number, Address, Nationality
            - Education details (degree, institution, duration)
            - Work Experience details (company, position, duration, responsibilities)

        ### Inputs
        Resume Text: {resume_text}
        Job Description Text: {job_description}

        ### Scoring Guidelines
        - Start at 0 and add points based on matches
        - Skills (0-40): +10 per matching skill, max 40
        - Experience (0-30): +5 per relevant year/role, max 30
        - Qualifications (0-20): +10 per matching degree/cert, max 20
        - Keywords (0-10): +2 per specific JD term, max 10
        - Deduct 10-20 points for critical missing requirements

        ### Output Format
        Return as a valid JSON string:
        ```json
        {{
            "Candidate Name": "",
            "Mail ID": "",
            "Phone Number": "",
            "Address": "",
            "Job Role": "",
            "Nationality": "",
            "Contract Type": "",
            "Education": [{{"degree": "", "institution": "", "duration": ""}}],
            "Work Experience": [{{"company": "", "position": "", "duration": "", "keyResponsibilities": [""]}}],
            "ATS Score": 0,
            "Missing Keywords": [],
            "Profile Summary": {{
                "Summary": "",
                "Scoring Breakdown": {{
                    "Skills": 0,
                    "Experience": 0,
                    "Qualifications": 0,
                    "Keywords": 0,
                    "Deductions": 0
                }}
            }}
        }}
        ```

        ### Notes
        - Return valid JSON only
        - Use exact details from inputs without assumptions
        - Vary scores based on actual matches (do not default to 60)
        - Provide specific breakdown numbers
        """

        try:
            response = self.client.chat.completions.create(
                model="mixtral-8x7b-32768",
                messages=[
                    {"role": "system", "content": "You are an ATS scoring assistant"},
                    {"role": "user", "content": prompt}
                ],
                max_tokens=2000,
                temperature=0.5  # Lowered for more consistent scoring
            )

            response_text = response.choices[0].message.content.strip()
            print(f"Raw LLM Response for resume: {response_text}")  # Debug output

            try:
                result = json.loads(response_text)
                if self.validate_json_response(result):
                    # Ensure ATS Score matches breakdown
                    breakdown = result["Profile Summary"]["Scoring Breakdown"]
                    calculated_score = (breakdown.get("Skills", 0) +
                                     breakdown.get("Experience", 0) +
                                     breakdown.get("Qualifications", 0) +
                                     breakdown.get("Keywords", 0) -
                                     breakdown.get("Deductions", 0))
                    if result["ATS Score"] != calculated_score:
                        print(f"Warning: ATS Score {result['ATS Score']} doesn't match breakdown {calculated_score}")
                    return result
                else:
                    return self._error_response("Invalid JSON structure from LLM", response_text)
            except json.JSONDecodeError:
                json_match = re.search(r'```json\s*([\s\S]*?)\s*```', response_text)
                if json_match:
                    try:
                        result = json.loads(json_match.group(1))
                        if self.validate_json_response(result):
                            return result
                    except json.JSONDecodeError:
                        pass
                return self._error_response("Failed to parse LLM response as JSON", response_text)

        except Exception as e:
            return self._error_response(f"API error: {str(e)}")

    def _error_response(self, message, raw_response=""):
        """Return a consistent error response format"""
        return {
            "Candidate Name": "",
            "Mail ID": "",
            "Phone Number": "",
            "Address": "",
            "Job Role": "",
            "Nationality": "",
            "Contract Type": "",
            "Education": [],
            "Work Experience": [],
            "ATS Score": 0,
            "Missing Keywords": [],
            "Profile Summary": {
                "Summary": message,
                "Scoring Breakdown": {
                    "Skills": 0,
                    "Experience": 0,
                    "Qualifications": 0,
                    "Keywords": 0,
                    "Deductions": 0,
                    "Error": message,
                    "Raw Response": raw_response[:500] if raw_response else "No response"
                }
            }
        }


In [11]:
def main():
    ats_scorer = ATSScorer(api_key=sec_key)

    resume_folder = "/content/Resumes"
    job_description = """
   # AI and ML Engineer

    ## About the Role
    We are seeking an experienced AI and ML Engineer to develop and implement AI and ML models.

    ## Responsibilities
    - Design and develop AI and ML models using Python
    - Implement Deep Learning and NLP techniques
    - Collaborate with cross-functional teams

    ## Required Qualifications
    - 3 years of experience in AI and ML
    - Bachelor's or Master's degree in Computer Science
    - Proficiency in:
      - Python
      - AI & Machine Learning
      - Deep Learning
      - NLP
      - Computer Vision
      - Generative AI

    ## Nice to Have
    - Experience with cloud-based technologies

    ## What We Offer
    - Competitive salary and benefits
    - Opportunities for growth and development
    - Collaborative and dynamic work environment
    - Flexible working hours


   """

    resume_texts = ats_scorer.process_resume_folder(resume_folder)
    results = {}
    for filename, resume_text in resume_texts.items():
        if resume_text:
            result = ats_scorer.calculate_ats_score(resume_text, job_description)
            results[filename] = result
        else:
            results[filename] = ats_scorer._error_response(f"Failed to extract text from {filename}")

    print(json.dumps(results, indent=2))

if __name__ == "__main__":
    main()

Raw LLM Response for resume: Here is the ATS score for the provided resume:

```json
{
    "Candidate Name": "Karen Santos",
    "Mail ID": "k.santos@email.com",
    "Phone Number": "123 456-7890",
    "Address": "Brooklyn, NY",
    "Job Role": "Senior Front-End Developer",
    "Nationality": "",
    "Contract Type": "",
    "Education": [{"degree": "Bachelor of Science", "institution": "Rutgers University", "duration": "2008 - 2012"}],
    "Work Experience": [
        {
            "company": "DTX",
            "position": "Front-End Developer",
            "duration": "September 2012 - Jan 2017",
            "keyResponsibilities": [
                "Participated in the initial wave of developers learning and implementing the React.JS library",
                "Rewrote HTML to meet industry and company standards for SEO and Accessibility",
                "Tested, debugged, and shipped 10s of 1000s of lines of code to various development teams",
                "Introduced Kanban Boar

In [12]:
def main():
    ats_scorer = ATSScorer(api_key=sec_key)
    # ats_scorer = ATSScorer(api_key="your-api-key-here")  # Replace with actual key

    resume_folder = "/content/Resumes"  # Update to your folder path
    job_description = """
      # Senior Frontend Developer

      ## About the Role
      We are looking for a skilled Senior Frontend Developer to lead the development of modern, scalable, and high-performing web applications. You will collaborate with designers, backend engineers, and product managers to create seamless user experiences.

      ## Responsibilities
      - Develop and maintain responsive, user-friendly web applications
      - Write clean, efficient, and maintainable code using modern frontend technologies
      - Optimize applications for performance, scalability, and security
      - Collaborate with UI/UX designers to implement visually appealing interfaces
      - Work closely with backend developers to integrate APIs and services
      - Stay updated with the latest frontend trends, tools, and best practices
      - Mentor junior developers and contribute to code reviews

      ## Required Qualifications
      - 5+ years of experience in frontend development
      - Strong proficiency in:
        - HTML, CSS, and JavaScript (ES6+)
        - React.js, Angular, or Vue.js
        - State management libraries (Redux, Context API, Vuex, etc.)
      - Experience with:
        - RESTful APIs and GraphQL integration
        - Performance optimization and cross-browser compatibility
        - Version control systems (Git, GitHub, GitLab, or Bitbucket)
      - Strong understanding of web accessibility and responsive design

      ## Nice to Have
      - Experience with TypeScript
      - Knowledge of CI/CD pipelines and DevOps practices
      - Familiarity with cloud-based platforms (AWS, Azure, or Firebase)
      - Exposure to backend technologies like Node.js or GraphQL

      ## What We Offer
      - Competitive salary and benefits
      - Opportunities for professional growth and leadership
      - Collaborative and fast-paced work environment
      - Flexible working hours and remote work options

   """

    resume_texts = ats_scorer.process_resume_folder(resume_folder)
    results = {}
    for filename, resume_text in resume_texts.items():
        if resume_text:
            result = ats_scorer.calculate_ats_score(resume_text, job_description)
            results[filename] = result
        else:
            results[filename] = ats_scorer._error_response(f"Failed to extract text from {filename}")

    print(json.dumps(results, indent=2))

if __name__ == "__main__":
    main()

Raw LLM Response for resume: Here is the ATS score for the provided resume and job description:

```json
{
    "Candidate Name": "Karen Santos",
    "Mail ID": "k.santos@email.com",
    "Phone Number": "123 456-7890",
    "Address": "Brooklyn, NY",
    "Job Role": "Senior Frontend Developer",
    "Nationality": "",
    "Contract Type": "",
    "Education": [{"degree": "Bachelor of Science", "institution": "Rutgers University", "duration": "2008 - 2012"}],
    "Work Experience": [
        {
            "company": "DTX",
            "position": "Front-End Developer",
            "duration": "September 2012 - Jan 2017",
            "keyResponsibilities": [
                "Participated in the initial wave of developers learning and implementing the React.JS library",
                "Rewrote HTML to meet industry and company standards for SEO and Accessibility",
                "Tested, debugged, and shipped 10s of 1000s of lines of code to various development teams",
                "Thi